# 04. Evaluation, Bias, and Deployment

Evaluation protocol:
- Locked holdout: 19,980 `srch_id` queries, group-disjoint from train, sha256 prefix `bc8ea6f6`.
- Local metric: NDCG@5 with the IDCG=0 = 0 convention; on this holdout no query has IDCG=0.
- Kaggle public score: 25 percent of the test set, used only as an external sanity check, never as a tuning target.

## Local versus public scores

| submission | role | local holdout | Kaggle public |
|---|---|---:|---:|
| `submit_anchor_3seed_avg.csv` | safe baseline | 0.395289 | 0.39500 |
| `submit_iter06_diverse_ensemble_blend.csv` | diverse ensemble | 0.399638 | 0.39685 |
| `submit_iter07_prior_3seed.csv` | final submission | 0.400682 | **0.39690** |

Local-to-public gap grew from +0.000289 (safe baseline) to -0.002788 (diverse ensemble) to -0.003782 (final). Public score did improve from 0.39500 to 0.39690, but the local gains overstated public gains. We attribute this to repeated holdout-based selection across the successive model variants (each stage chose the model that scored best on the same locked holdout), which inflates expected holdout NDCG even when each individual decision was leakage-safe.

## Anti-overfitting decision in the final exploratory stage

A final blend search included a weighting (0.60 final / 0.30 diverse / 0.10 safe) that scored 0.401482 on the holdout. We did not select it: it was the holdout-max under the same-set selection, and selecting it would have continued the same-set selection pattern. Instead we chose a more conservative weighting per a pre-registered heuristic and ultimately did not upload that blend. The headline submission is the simpler 3-seed historical-prior model.

## Bias detection

We segment the holdout along two dimensions and report NDCG@5 plus exposure@5 (share of within-search positives that surface in the top 5).

Top-K visitor countries (top 5 plus a tail bucket for queries with too-few rows per individual country):

| `prop_country_id` | NDCG@5 | exposure@5 |
|---|---:|---:|
| 219 | 0.403996 | 0.9980 |
| 100 | 0.415553 | 0.9932 |
|  55 | 0.392032 | 1.0000 |
|  31 | 0.366490 | 0.9997 |
|  99 | 0.349878 | 0.9612 |
| tail | 0.355789 | 0.9980 |

Property popularity tertile (by total impressions on train):

| tertile | NDCG@5 | exposure@5 |
|---|---:|---:|
| high   | 0.390424 | 0.9266 |
| medium | 0.395074 | 0.2651 |
| low    | 0.426924 | 0.2010 |

Interpretation: high-popularity hotels dominate the top-5 (exposure 0.93). Low-popularity hotels rarely appear in the top-5 even when they are relevant in their own searches. We treat this as exposure disparity rather than discrimination, since the per-search NDCG of low-popularity hotels is high when they do appear.

## Mitigation: post-processing exposure rerank

We compute a tertile prior $P_{\text{tertile}}$ from the train set and rerank with
$$
\tilde{s}(p) = s(p) + \lambda \cdot \bigl(-\log P_{\text{tertile}(p)}\bigr),
$$
with $\lambda = 0.20$. This is a deployment variant; the primary Kaggle submission is unmodified. With $\lambda = 0.20$:

| metric | before | after |
|---|---:|---:|
| global NDCG@5 | reference | reference - 0.003090 |
| exposure@5, high tertile | 0.9266 | 0.8965 |
| exposure@5, medium tertile | 0.2651 | 0.3184 |
| exposure@5, low tertile | 0.2010 | 0.2227 |

We give up about 0.31 percentage points of NDCG@5 globally to bring medium and low tertile exposure up by roughly 5 and 2 percentage points respectively.

![bias exposure](../report/figures/bias_exposure_before_after.png)

## Deployment monitoring

What an operations team would watch:
- NDCG@5 by visitor country segment and by property popularity tertile, monthly.
- Exposure@5 by tertile, weekly, with alerting if the high-tertile share rises above pre-mitigation levels.
- Cold-start hotels (no train history): served with the global-rate fallback in the prior features. This is by construction (Laplace smoothing) and we do not need a separate cold-start path.
- Feature drift on `prop_location_score2`, `price_usd`, and the prior features themselves; we keep a fixed reference distribution from the training slice.
- Model retraining cadence is offline, monthly, with the same group-aware split protocol.